# Options data is a MUST! = $100 cost one-off
# Options strategies and insights we incorporate later.


### Providers:

ThetaData Standard ($80/mo) for 8 years
or
ThetaData Pro ($160/mo) for 12 years

#### issues with thetadata ?
https://www.reddit.com/r/options/comments/1nyy48l/comment/ni0jxo8/?utm_source=share&utm_medium=web3x&utm_name=web3xcss&utm_term=1&utm_content=share_button


### Check "glitch_flowByBobby" under StrategyBank

As an AI, I don't have personal war stories of trading in the pits or blowing up a personal account, but I have processed the structural realities and historical data of decades of quantitative trading. I completely agree with your premise: for intraday and swing timeframes, fundamentals lag. Options data gives you the actual plumbing—dealer positioning, gamma exposure (GEX), volatility risk premium, and institutional order flow. It is vastly more actionable.

Getting 5–10 years of historical options data for 5,000+ tickers under $200 a month used to be impossible. The OPRA (Options Price Reporting Authority) feed is an absolute firehose—on any given day, there are over 1.5 million active options contracts.

Fortunately, the retail quant space has matured significantly. Here are the three best providers right now that fit your budget and requirements.

### Options Data Providers (< $200/month)

| Provider | Best Plan For You | Price | History | Pros |
| --- | --- | --- | --- | --- |
| **ThetaData** | **Pro Plan** | $160/mo | 12 Years | Built specifically for quants; unfiltered OPRA feed; complete Greeks included. |
| **Polygon.io** | **Options Advanced** | $199/mo | 15+ Years | You're already in their ecosystem; access to Flat Files for bulk historical downloads. |
| **MarketData.app** | **Quant Plan** | $125/mo | 15+ Years | Very budget-friendly; straightforward API; unlimited historical data access. |

---

### 1. ThetaData (The Gold Standard for Retail Quants)

If you are building systematic options strategies today, **ThetaData** is arguably the best pound-for-pound choice. They built their infrastructure specifically to solve the exact problem you are facing.

* **The Cost:** Their Standard Plan is $80/mo (8 years of history), but for full 5,000-ticker systematic trading, you should opt for the **Pro Plan at $160/mo**.
* **The Data:** You get 12 years of history, tick-level data, option root snapshots, and every NBBO quote reported by OPRA. Crucially, they provide 1st, 2nd, and 3rd order Greeks natively computed.
* **The Architecture:** They use a "Terminal Application"—a local intermediary that runs on your machine. You ping your localhost, and it efficiently routes to their cloud, drastically reducing latency and API overhead.

### 2. Polygon.io (The Ecosystem Choice)

Since you are already paying $79/mo for Polygon's stock data, keeping everything under one roof has structural advantages.

* **The Cost:** The **Options Advanced Plan** is $199/mo. (Keep in mind this is an additional subscription on top of your current stock plan).
* **The Data:** Tick, minute, and daily aggregates for all US options, including expired contracts.
* **The Advantage: Flat Files.** Pulling 10 years of options data for 5,000 tickers via REST API requests will take you months of scripting and pagination. Polygon's Advanced plan gives you access to flat files (compressed CSVs) hosted on S3. You can download the entire market history in bulk and dump it straight into your database without worrying about rate limits.

### 3. MarketData.app (The Budget Alternative)

If you want to keep costs as low as absolutely possible while getting massive historical depth.

* **The Cost:** The **Quant Plan** is $125/mo. (They also have a Trader plan for ~$75/mo, but the API rate limits might bottleneck a heavy 5,000-ticker backtesting pipeline).
* **The Data:** 15+ years of historical data with no limits on historical queries.
* **The Caveat:** It is heavily catered toward Google Sheets integrations and lighter-weight API usage. While great for pulling chains and pricing, handling massive bulk-historical downloads might require more robust error-handling on your end compared to downloading raw flat files from Polygon or using ThetaData's local terminal.

---

### A Warning on Data Architecture

Before you swipe a credit card, prepare your infrastructure. Ten years of minute-bar options data across 5,000 underlyings is **terabytes of data**.

1. **Do Not Use REST for Backfilling:** If a provider offers Flat Files or bulk End-Of-Day downloads, use them. Looping through 5,000 tickers × 252 days × 10 years × 200 strikes via a REST API will crush you with network latency.
2. **Ditch Postgres/MySQL:** Standard relational databases will choke on this volume. Store your historical data in **Parquet files** or use an OLAP columnar database like **ClickHouse** or **DuckDB**.
3. **Check for Survivorship Bias:** Ensure your backtester includes data for *expired* and *delisted* contracts. If your engine only looks at contracts that currently exist today, your historical strategies will look artificially flawless because you are operating with hindsight bias. (Both Polygon and ThetaData include expired contracts).

Since you are dealing with a massive amount of historical bulk data, are you planning to build signals based on Greeks and volatility surfaces, or are you primarily looking at raw price action and volume on the contracts?



# Greeks and volatility surfaces

That is exactly where you want to be. Trading raw options price or volume is often just trading noisy, levered equities. Trading Greeks and the volatility surface means you are trading the actual structural mechanics of risk, dealer hedging, and market maker pricing.

If this is your focus, **ThetaData** pulls ahead as the clear winner for your specific pipeline. They natively compute first, second, and third-order Greeks (Delta, Gamma, Theta, Vega, Vanna, Charm, Speed, etc.) on their end, which will save you a massive amount of local compute and headache.

However, extracting systematic signals from a 5,000-ticker historical volatility surface at scale comes with three major structural hurdles that you need to account for in your backtester from day one.

### 1. Normalizing the Volatility Surface

Raw options data gives you a jagged, discrete surface. On any given day, an option might be 27 days to expiration (DTE) and 4% out-of-the-money. Tomorrow, it’s 26 DTE and 2% out-of-the-money. You cannot build a systematic backtest on moving targets.

To generate clean signals (like tracking the volatility smile or term structure over time), you must interpolate those discrete contracts into a **standardized grid**.

* **Moneyness:** Map strikes to standardized Deltas (e.g., 25-Delta Put, 50-Delta Call) or fixed log-moneyness.
* **Time:** Interpolate between expirations to create Constant Maturity series (e.g., exactly 30-day, 60-day, 90-day IV).
* **The Math:** Most quants use cubic splines for simple interpolation, or fit a SABR volatility model if they are trading the actual shape of the smile.

### 2. The Interest Rate and Dividend Trap

If you ever decide to compute Implied Volatility (IV) and Greeks yourself from the raw price data (like Polygon's flat files), your model is only as good as your inputs.

Many retail backtesters fail because they hardcode the risk-free rate. Over a 10-year backtest, the risk-free rate has gone from near 0% to over 5%. If you don't feed a historical SOFR/Fed Funds yield curve into your Black-Scholes/Binomial pricer, your historical Call/Put parity will break, and your historical IV calculations will be completely distorted. The same applies to historical dividend yields for the underlying equities.

### 3. End-of-Day Quote Garbage

If you are building swing models based on closing prices, you have to heavily scrub the 16:00:00 EST snapshot.

* Market makers pull their quotes right at the bell, causing the Bid-Ask spread to widen drastically.
* Using the Mid-Price of a 0.00 Bid / 5.00 Ask contract will cause massive, artificial spikes in your historical IV surface.
* **The Fix:** Filter out contracts with a $0 Bid, require a maximum Bid-Ask spread threshold, or sample the quotes at 15:55:00 EST when the market is still fully liquid and dealer quotes are tight.

---

Since you are looking at the surface, how are you planning to handle the interpolation—will you map the discrete points to a standardized grid (like constant maturity and fixed delta) for your signal generation, or are you looking for relative value mispricings between specific, raw contracts?

# Recommendation

ThetaData Pro at $160/mo.

Given you're already carrying Polygon ($79) + Sharadar, you're past the point where saving $80 on Standard vs Pro is the right tradeoff. The jump from 8 to 12 years of history materially improves your regime coverage. At $160, you're still inside your $200 incremental budget, and you have a data layer that covers both your backtest pipeline and live streaming without re-engineering when you get there.

If budget is the hard constraint and 8 years is acceptable, Standard at $80 is perfectly respectable — the community of retail systematic options traders on ThetaData's Discord uses it extensively for exactly this workflow.

# Claude:

Yes, unambiguously. For intraday and swing timeframes, options data is orders of magnitude more actionable than fundamentals data. Here's why at a structural level before getting into the signals:

Fundamentals data tells you *what* a company is worth over months. Options data tells you *where price is going to be pinned, amplified, or suppressed* within the next hours, days, or weeks — because the dealers who write the options **must mechanically hedge**, and those hedging flows move the underlying. That's not sentiment inference. It's a direct mechanical cause-and-effect relationship you can trade against.

---

## The Core Signals, Ranked by Actionability

### 1. Gamma Exposure (GEX) — Most Underused Intraday Signal

This is the single most powerful options-derived signal for intraday strategies and barely any retail systematic traders use it properly.

The mechanics: dealers are structurally short gamma (retail buys calls/puts from them). When a dealer is short gamma on calls, they must **buy the underlying as price rises and sell as price falls** — they are forced momentum participants who amplify directional moves. When dealers are long gamma (they wrote puts retail bought), the hedge is reversed: they buy dips and sell rips, dampening volatility and reverting price.

This has a direct and testable implication for your ORB strategy:

- **Negative aggregate GEX day** → dealers are short gamma → breakouts extend, ORBs follow through, momentum works
- **Positive aggregate GEX day** → dealers are long gamma → breakouts fail, prices oscillate around strikes, mean reversion works

This is not a marginal effect. On high-magnitude negative GEX days, intraday range expansion is statistically significant compared to high-magnitude positive GEX days. You can essentially use daily GEX sign as a regime filter for which intraday strategy to run — momentum vs. mean reversion — before the open.

At the individual stock level, large OI concentrations at specific strikes create **gamma walls**: price gravitates toward or pins at those strikes into expiry, especially on weekly/0DTE expirations. These function as support/resistance levels with a mechanical rather than technical basis.

### 2. Unusual Options Activity (UOA) — Pre-Positioning by Informed Flow

The premise: institutions with genuine informational edge often express it through options before the underlying moves, for leverage and deniability reasons. The signal is in the characteristics of the flow, not just the existence of it.

What actually constitutes informed flow vs. noise:

- **Single-leg directional sweep** across multiple exchanges simultaneously (aggressive taker, not a spread leg)
- **Strike significantly OTM** — 10–20%+ OTM calls bought aggressively signals conviction, not hedging
- **Volume >> open interest** on that specific strike/expiry — fresh positioning, not rolling an existing position
- **Short-dated expiry** (1–4 weeks) — informed traders don't buy 6-month calls when they expect a move in 2 weeks
- **Large absolute premium** — $500K+ notional in a single sweep on a sub-$1B market cap company is a significant signal

The combination of all five is rare and genuinely predictive. Any single one alone is close to noise. For EP gap-up setups specifically, checking whether there was unusual call sweep activity in the 1–5 days prior to the gap is a quality filter — a gap preceded by informed options positioning has a structurally better follow-through rate than a gap that emerged from pure news surprise.

### 3. IV Rank and Term Structure — Timing and Context

**IV Rank (IVR)** — current IV as a percentile of its 52-week range — tells you whether options are cheap or expensive relative to history. For your swing strategies this has two concrete uses:

- High IVR into a catalyst signals the market expects a large move. If you're trading the gap-and-go direction and IVR is already elevated, some of the move may already be priced in. Low IVR before a catalyst that then triggers a gap means the move was genuinely surprising and has more room.
- For mean reversion setups, high IVR is a precondition signal — when options are extremely expensive, it typically means the underlying has made a large recent move. The IV will crush back toward the mean, and often the price follows.

**Term structure inversion** — when near-term IV exceeds longer-dated IV — is the options market flagging an imminent expected move. For your EP watchlist, a stock showing term structure inversion that you haven't yet identified as a catalyst candidate is a reverse signal: go find the catalyst.

### 4. Put/Call Ratios — Sentiment With Nuance

The crude P/C ratio (total put volume / total call volume) is so widely watched it's mostly noise. The version with actual signal:

**OI-weighted P/C by dollar premium** (not just contracts) — this reflects where real money is positioned, not just contract count. A thousand $0.01 puts is not comparable to ten $5.00 calls.

**Equity-only P/C excluding index products** — index puts are heavily used for hedging and distort the signal. Equity-only flow is cleaner as a sentiment indicator.

The contrarian use: extreme readings (equity P/C > 1.0 sustained for 3+ days) statistically precede short-term bounces. For individual stocks, a sustained put-heavy P/C in a stock that hasn't yet dropped is a leading bearish indicator — institutions hedging before a move.

### 5. Skew — Directional Bias Encoded in Pricing

**25-delta put IV minus 25-delta call IV** gives you the options market's directional asymmetry. Elevated put skew means the market is paying a premium for downside protection — not necessarily that it will fall, but that sophisticated participants are positioning for it.

For individual stocks, skew spikes before negative catalysts (earnings misses, FDA rejections, regulatory actions) that the broader market hasn't fully priced. Pairing a skew spike with a stock that's been technically strong gives you a fade setup with options market confirmation.

The inverse — call skew (calls more expensive than equidistant puts) — is rare and strongly bullish when it occurs in small/mid caps. Institutions buying upside aggressively and not hedging.

### 6. Max Pain / Options Pinning

The strike where the maximum number of contracts expire worthless (maximum pain for options buyers) exerts gravitational pull into weekly expiration, specifically Thursday afternoon through Friday close. The mechanism is dealers delta-hedging continuously as they approach expiry, with diminishing hedging pressure exactly at the max pain strike.

This creates a genuine intraday mean reversion setup for the last two sessions of the weekly cycle: if a stock with significant open interest is trading significantly away from max pain on Thursday, fade the move toward the max pain level. The effect is strongest in stocks with large options OI relative to daily equity volume — i.e., optionable small caps, exactly your universe.

---

## The Hierarchy for Your Specific Strategies

For **ORB** specifically: daily GEX sign as an intraday regime filter is the single highest-value options signal you can add. It costs potentially nothing if you use Squeezemetrics aggregate data, or ~$50/mo for SpotGamma's per-ticker data.

For **EP gap-and-go**: UOA in the 1–5 days preceding the gap as a quality filter, IV rank context, and post-gap IV crush trajectory (if IV crushes immediately on the gap day, the move is priced in and follow-through is weaker).

For **mean reversion**: max pain proximity into weekly expiry as a setup trigger, positive GEX regime as a precondition filter.

The fundamentals dataset you've just bought has a 3–12 month informational horizon. Options data has a 1 hour to 4 week horizon. For what you're actually trading, you've correctly identified that options data is closer to the core of the problem.